In [1]:
from dotenv import load_dotenv
import os
load_dotenv(".env")

True

In [2]:
from typing import Literal
from datetime import datetime
from pydantic import BaseModel
from langchain_core.tools import tool

In [3]:
@tool
def write_email(to:str,subject:str,content:str)->str:
    """write and send an email"""
    return f"Email sent to {to} with subject '{subject}' and content: {content}"

@tool
def schedule_meeting( attendees: list[str], subject: str, duration_minutes: int, preferred_day: datetime, start_time: int
)->str:
    """schedule a calendar meeting"""
    date_str = preferred_day.strftime("%A, %B %d, %Y")
    return f"Meeting '{subject}' scheduled on {date_str} at {start_time} for {duration_minutes} minutes with {len(attendees)} attendees"


@tool
def check_calendar_availability(day: str) -> str:
    """Check calendar availability for a given day."""
    # Placeholder response - in real app would check actual calendar
    return f"Available times on {day}: 9:00 AM, 2:00 PM, 4:00 PM"

@tool
class Done(BaseModel):
      """E-mail has been sent."""
      done: bool

In [30]:
from langgraph.graph import MessagesState
class State(MessagesState):
    email_input: dict
    classification_decision: Literal["ignore", "respond", "notify"]

In [5]:
%load_ext autoreload
%autoreload 2

from pydantic import BaseModel, Field
from email_assistant.utils import parse_email, format_email_markdown
from email_assistant.prompts import triage_system_prompt, triage_user_prompt, default_triage_instructions, default_background
from langchain.chat_models import init_chat_model
from langgraph.graph import END
from langgraph.types import Command

In [6]:
from rich.markdown import Markdown
Markdown(triage_system_prompt)

You are an assistant that triages incoming email messages.

In [7]:
Markdown(triage_user_prompt)

Decide whether the email should be ignored, responded to, or forwarded as a notification.

In [8]:
Markdown(default_background)

You are working in an email assistant application that helps manage and classify email. Use the provided message   
metadata and body to decide the best action.

In [9]:
Markdown(default_triage_instructions()) 

Read the email content and choose one of: ignore, respond, notify. If the email clearly requires a reply, choose   
respond. If it is informational or not urgent, choose ignore. If it needs a human or team notification, choose     
notify.

In [10]:
from langchain_groq import ChatGroq

In [20]:
from langchain_community.llms import Ollama
# from langchain.prompts import PromptTemplate
from langchain.chat_models import init_chat_model

In [31]:
class RouterSchema(BaseModel):
    """Analyze the unread email and route it according to its content."""

    reasoning: str = Field(
        description="Step-by-step reasoning behind the classification."
    )
    classification: Literal["ignore", "respond", "notify"] = Field(
        description="The classification of an email: 'ignore' for irrelevant emails, "
        "'notify' for important information that doesn't need a response, "
        "'respond' for emails that need a reply",
    )

# Initialize the LLM for use with router / structured output
llm = init_chat_model(model = "gemma4:e4b",
                      model_provider="ollama", temperature=0.0)
llm_router = llm.with_structured_output(RouterSchema) 

def triage_router(state: State) -> Command[Literal["response_agent", "__end__"]]:
    """Analyze email content to decide if we should respond, notify, or ignore."""
    
    author, to, subject, email_thread = parse_email(state["email_input"])
    system_prompt = triage_system_prompt.format(
        background=default_background,
        triage_instructions=default_triage_instructions
    )

    user_prompt = triage_user_prompt.format(
        author=author, to=to, subject=subject, email_thread=email_thread
    )

    result = llm_router.invoke(
        [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]
    )
    
    if result.classification == "respond":
        print("📧 Classification: RESPOND - This email requires a response")
        goto = "response_agent"
        update = {
            "messages": [
                {
                    "role": "user",
                    "content": f"Respond to the email: \n\n{format_email_markdown(subject, author, to, email_thread)}",
                }
            ],
            "classification_decision": result.classification,
        }
        
    elif result.classification == "ignore":
        print(" Classification: IGNORE - This email can be safely ignored")
        goto = END
        update =  {
            "classification_decision": result.classification,
        }
        
    elif result.classification == "notify":
        print(" Classification: NOTIFY - This email contains important information")
        # For now, we go to END. But we will add to this later!
        goto = END
        update = {
            "classification_decision": result.classification,
        }
        
    else:
        raise ValueError(f"Invalid classification: {result.classification}")
    return Command(goto=goto, update=update)